#  Step 1: Load the data

In [2]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')

DRIVE     = Path('/content/drive/MyDrive/m5_project')
PROCESSED = DRIVE / 'processed'

# split boundaries (day numbers), same as the paper's M5 setup
TRAIN_END  = 1885      # last training day
VAL_START  = 1886      # validation: d_1886 - d_1913
TEST_START = 1914      # test: d_1914 - d_1941 (scored once, at the very end)
HORIZON    = 28

df = pd.read_parquet(PROCESSED / 'ca12_long.parquet',
                     columns=['id', 'item_id', 'dept_id', 'cat_id', 'store_id',
                              'd', 'sales', 'sell_price'])
df['sales'] = df['sales'].astype('int32')     # int16 would overflow when we add up levels

print(df.shape)                                # expect (6096944, 8)
print('days:', df.d.min(), '->', df.d.max())   # expect 847 -> 1941

Mounted at /content/drive
(6096944, 8)
days: 847 -> 1941


# Step 2: Define 8 levels

- Defines how item-store sales are added up into higher levels (the "tree").

In [12]:
LEVELS = {
    'L1_total':      [],
    'L2_store':      ['store_id'],
    'L3_cat':        ['cat_id'],
    'L4_dept':       ['dept_id'],
    'L5_store_cat':  ['store_id', 'cat_id'],
    'L6_store_dept': ['store_id', 'dept_id'],
    'L7_item':       ['item_id'],
    'L8_item_store': ['item_id', 'store_id'],
}

# teacher levels for H-Pro-Avg: (L1-L4)
TEACHER_LEVELS = ['L1_total', 'L2_store', 'L3_cat', 'L4_dept']
for name, cols in LEVELS.items():
    n = 1 if not cols else df[cols].drop_duplicates().shape[0]
    print(f'{name:<15} {n:>5} series')

L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series



| Level | Count | Meaning |
|---|---|---|
| L1 Total | 1 | both stores, all products added together |
| L2 Store | 2 | one series per store (CA_1, CA_2) |
| L3 Category | 3 | FOODS, HOBBIES, HOUSEHOLD |
| L4 Department | 7 | e.g. FOODS_1, FOODS_2, FOODS_3 ... |
| L5 Store × Category | 6 | 2 stores × 3 categories |
| L6 Store × Department | 14 | 2 stores × 7 departments |
| L7 Item | 3,049 | each product, both stores added |
| L8 Item × Store | 6,098 | each product in each store (bottom level) |

# Step 3: The bottom table (one row per product-store)
- Right now the data is long (one row per product per day).
- To add things up quickly, we turn it into a grid: one row per product-store and one column per day. Every level is built from this grid.
- 6,098 × 1,095 grid; total units unchanged

In [4]:
# grid: rows = 6,098 product-stores, columns = days, values = units sold
bottom = df.pivot(index='id', columns='d', values='sales').fillna(0).astype('int32')

# for each product-store: its item, dept, category and store
meta = (df[['id', 'item_id', 'dept_id', 'cat_id', 'store_id']].drop_duplicates('id').set_index('id').astype(str))
meta = meta.loc[bottom.index]
bottom.index = bottom.index.astype(str)
print(bottom.shape)                    # expect (6098, 1095)
assert bottom.values.sum() == df.sales.sum(), 'units lost while building grid'


# Output
# 6,098 rows, one for each product in each store
# 1,095 columns, one for each day from d_847 to d_1941

(6098, 1095)


# Step 4: Add up every level
- This builds the tree. Each level’s daily sales = the bottom rows added up by that level’s columns.

In [5]:
def aggregate(level_cols):
    """Add up bottom rows by the given columns ([] = everything)."""
    if not level_cols:
        return bottom.sum().to_frame('Total').T
    return bottom.join(meta[level_cols]).groupby(level_cols).sum()

# build all 8 levels
sales_by_level = {name: aggregate(cols) for name, cols in LEVELS.items()}

# check: every level must add up to the same grand total
grand_total = bottom.values.sum()
for name, table in sales_by_level.items():
    assert table.values.sum() == grand_total, f'{name} does not add up'
    print(f'{name:<15} {table.shape[0]:>5} series x {table.shape[1]} days')


L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days


In [6]:
print("Level 1: Sum of both the stores")
print(sales_by_level['L1_total'].iloc[:, :5])    # 1 row: both stores together
print("----------------------------------------")

print("\n Level 2: Sum of each Store")
print(sales_by_level['L2_store'].iloc[:, :5])    # 2 rows: CA_1, CA_2
print("----------------------------------------")

print("\nLevel 3: Sum of each Category")
print(sales_by_level['L3_cat'].iloc[:, :5])      # 3 rows: FOODS, HOBBIES, HOUSEHOLD
print("----------------------------------------")

print("\nLevel 4: Sum of each Department")
print(sales_by_level['L4_dept'].iloc[:, :5])     # 7 rows: departments

Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
----------------------------------------

 Level 2: Sum of each Store
           847   848   849   850   851
store_id                              
CA_1      4390  4766  4416  4390  3575
CA_2      3355  3981  3580  3807  2691
----------------------------------------

Level 3: Sum of each Category
            847   848   849   850   851
cat_id                                 
FOODS      4935  5481  5049  5228  3818
HOBBIES     932  1104   925   775   879
HOUSEHOLD  1878  2162  2022  2194  1569
----------------------------------------

Level 4: Sum of each Department
              847   848   849   850   851
dept_id                                  
FOODS_1      1059  1032   914   979   746
FOODS_2       481   523   514   641   480
FOODS_3      3395  3926  3621  3608  2592
HOBBIES_1     895  1048   865   725   853
HOBBIES_2      37    56    60    50    26
HOUSEHOLD_1  1342  1578  1478 

# Step 5: Scale, the bottom part of the paper’s RMSSE
- It measures how much each series normally jumps from one day to the next, so errors on calm and jumpy series can be compared fairly.

In [7]:
def get_scales(last_day):
    """RMSSE denominator: average squared day-to-day change, from first sale up to last_day."""
    scales = {}
    for name, table in sales_by_level.items():
        hist = table.loc[:, :last_day].to_numpy(dtype=float)   # history up to last_day only
        started = np.cumsum(hist > 0, axis=1) > 0               # Get when the sell started, i.e - sale was not 0
        sq_change = np.diff(hist, axis=1) ** 2                  # (today - yesterday)^2
        counted = started[:, :-1]                               # only count after the first sale
        n_days = np.maximum(counted.sum(axis=1), 1)
        scales[name] = (sq_change * counted).sum(axis=1) / n_days
    return scales

scales_val = get_scales(TRAIN_END)          # 1885: used when scoring the validation window
for name, s in scales_val.items():
    print(f'{name:<15} min {s.min():>12.2f}   max {s.max():>14.2f}   zeros {(s == 0).sum()}')

L1_total        min   2867627.72   max     2867627.72   zeros 0
L2_store        min    687850.13   max      859987.45   zeros 0
L3_cat          min     39013.87   max     1171468.08   zeros 0
L4_dept         min       717.30   max      664320.02   zeros 0
L5_store_cat    min     11441.37   max      419022.21   zeros 0
L6_store_dept   min       272.48   max      253573.74   zeros 0
L7_item         min         0.09   max        2617.81   zeros 0
L8_item_store   min         0.02   max        1759.61   zeros 0


# Step 6: Weights (how important is each series?)

- Gives every series a weight equal to its share of dollar sales
in the 28 days before the forecast. Within each level, the weights add up to 1.

In [8]:
# price grid, same rows and days as the sales grid
price = df.pivot(index='id', columns='d', values='sell_price').fillna(0)
price.index = price.index.astype(str)
price = price.loc[bottom.index]

def get_weights(last_day):
    """M5 weights: dollar sales in the 28 days up to last_day, as a share of the level's total."""
    window  = list(range(last_day - 27, last_day + 1))
    dollars = (bottom[window] * price[window]).sum(axis=1)       # $ per product-store
    weights = {}
    for name, cols in LEVELS.items():
        if not cols:                                             # L1: only one series
            weights[name] = pd.Series([1.0], index=sales_by_level[name].index)
            continue
        level_dollars = dollars.groupby([meta[c] for c in cols]).sum()
        weights[name] = (level_dollars / level_dollars.sum()).reindex(sales_by_level[name].index)
    return weights

weights_val = get_weights(TRAIN_END)        # 1885: weights for scoring validation
for name, w in weights_val.items():
    assert abs(w.sum() - 1) < 1e-9, f'{name} weights do not sum to 1'
    print(f'{name:<15} weights sum to {w.sum():.3f}   largest {w.max():.3f}   series {len(w)}')

L1_total        weights sum to 1.000   largest 1.000   series 1
L2_store        weights sum to 1.000   largest 0.516   series 2
L3_cat          weights sum to 1.000   largest 0.554   series 3
L4_dept         weights sum to 1.000   largest 0.349   series 7
L5_store_cat    weights sum to 1.000   largest 0.288   series 6
L6_store_dept   weights sum to 1.000   largest 0.190   series 14
L7_item         weights sum to 1.000   largest 0.008   series 3049
L8_item_store   weights sum to 1.000   largest 0.006   series 6098


# Step 7 - Scoring function

- Combine step 5 & step 6 providing weighted RMSSE at each level, then R_H as their average
- use to grade every forecast and choose every hyperparameter setting.

In [9]:
def roll_up(grid, level_cols):
    """Add up a bottom-level grid to one level (same rule as Step 4)."""
    if not level_cols:
        return grid.sum().to_frame('Total').T
    return grid.join(meta[level_cols]).groupby(level_cols).sum()


def score(forecast, last_day):
    """Paper metric: weighted RMSSE at each level, and R_H = average of the levels."""
    days    = forecast.columns
    scales  = get_scales(last_day)
    weights = get_weights(last_day)

    level_scores = {}
    for name, cols in LEVELS.items():
        actual = sales_by_level[name][days]          # true sales at this level
        pred   = roll_up(forecast, cols)             # forecast added up to this level
        mse    = ((actual - pred) ** 2).mean(axis=1) # average squared error per series
        rmsse  = np.sqrt(mse / scales[name])         # divide by scale, then square root
        level_scores[name] = (weights[name] * rmsse).sum()

    level_scores = pd.Series(level_scores)
    return level_scores, level_scores.mean()        # per-level scores, R_H

# Step 8 - Sanity check of the scorer
Two forecasts with known answers:
1. Perfect forecast (true sales) → R_H must be 0.
2. "Copy the last 28 training days" → R_H should be near 1

In [10]:
val_days = list(range(VAL_START, TEST_START))       # 1886 ... 1913 (28 days)

# Test 1: perfect forecast -> must score 0
perfect = bottom[val_days]
_, rh_perfect = score(perfect, TRAIN_END)
print('Perfect forecast R_H:', round(rh_perfect, 3))
assert rh_perfect == 0, 'scorer is broken'

# Test 2: copy the last 28 training days -> should be close to 1
naive = bottom.loc[:, TRAIN_END - 27 : TRAIN_END].copy()
naive.columns = val_days
naive_levels, rh_naive = score(naive, TRAIN_END)

print(naive_levels.round(3))
print('Copy-last-28-days R_H:', round(rh_naive, 3))


Perfect forecast R_H: 0.0
L1_total         0.577
L2_store         0.750
L3_cat           0.616
L4_dept          0.694
L5_store_cat     0.798
L6_store_dept    0.875
L7_item          1.149
L8_item_store    1.155
dtype: float64
Copy-last-28-days R_H: 0.827


In [11]:
assert round(rh_naive, 3) == 0.827, 'hierarchy output changed! check data or code'